In [ ]:
import pandas as pd
#from sqlalchemy import create_engine
import datetime as dt
import time as tm
import numpy as np
import json
import requests
#from pymongo import MongoClient
#import pymysql
from datetime import datetime,timedelta
#import matplotlib.pyplot as plt
import psycopg2
import ast

In [ ]:
import os
from config import *  # all DB / API handles come from environment variables, see config.py


In [ ]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
pd.set_option('display.max_rows', 5000)
pd.set_option('display.max_columns', 5000)
pd.set_option('display.width', 100000)



In [ ]:
toll_fare_nhai=pd.read_csv("Toll Plaza Fares - 2022 April - Toll_Rules_Temp_Table.csv")
toll_fare_nhai=toll_fare_nhai[~toll_fare_nhai['Plaza Name'].isna()]
toll_fare_nhai['Plaza Code']=toll_fare_nhai['Plaza Code'].apply(int).apply(str).apply(lambda x : x.zfill(6))

In [ ]:
txn=pd.read_sql("select json_extract_path_text(extra_data, 'plazaCode') as toll_id from fastag_transaction where category='TOLL_DEBIT' and created_at> 'July 30, 2022'   ",con )    



In [ ]:
toll_price_exists=pd.DataFrame(txn['toll_id'].apply(lambda x : x.zfill(6)).unique(),columns=['toll_id'])

In [ ]:
toll_price_exists['toll_fare_on_nhai']=toll_price_exists['toll_id'].apply(lambda x : 'yes' if x in toll_fare_nhai['Plaza Code'].to_list( ) else 'no' )   



In [ ]:
toll_price_exists.toll_fare_on_nhai.value_counts(normalize=True)

In [ ]:
toll_price_exists

In [ ]:
txn=pd.merge(txn,toll_price_exists,how='left',on='toll_id')



In [ ]:
txn['toll_fare_on_nhai'].value_counts(normalize=True)

In [ ]:
priority_on_needed_tolls=txn[txn['toll_fare_on_nhai']=='no'].toll_id.value_counts().keys().to_list()

In [ ]:
priority_on_needed_tolls

In [ ]:
toll_price_exists=toll_price_exists.rename(columns={'toll_fare_on_nhai':'toll_fare_on_nhai_site'})

In [ ]:
toll_price_exists=toll_price_exists.drop_duplicates('toll_id')

In [ ]:
tickets=pd.read_sql(
'''select 
created_at,updated_at,
id as ticket_id,vehicle_id,
issue_identified,txn_id,
issue_status,custom_source,
json_extract_path_text(ticket_fields, 'pobSubStatus') as pobSubStatus,
json_extract_path_text(ticket_fields, 'closedReason') as closedReason,
json_extract_path_text(ticket_fields, 'txnId') as txnTime,
json_extract_path_text(ticket_fields, 'ticketType') as QDR,
json_extract_path_text(ticket_fields, 'disputedRRN') as disp_rrn,
json_extract_path_text(ticket_fields, 'originalRRN') as orgn_rrn,
json_extract_path_text(ticket_fields, 'tollPlazaNameAndNumber') as plaza_code,
json_extract_path_text(ticket_fields, 'txnAmount') as txnAmount,
json_extract_path_text(ticket_fields, 'disputeAmount') as disputeAmount


from fleet_lake.escalations where 
created_at>='2022-06-01' and 
issue_identified in (
'I did not get benefit of less toll-charge for a return journey.',
'I have been charged more than normal',
'I have been charged twice.',
'I paid by cash & by Fastag both.',
'I was incorrectly charged extra as debit adjustment',
'Vehicle didn''t cross this toll plaza.',
'I did not get benefit of less toll-charge for an early exit from a toll road.')''',con
)

In [ ]:
auti_tickets=tickets[tickets['custom_source']=='Payments Service'][tickets['issue_status']=='Closed']

In [ ]:
auti_tickets=pd.merge(auti_tickets,toll_price_exists, how='left',left_on='plaza_code',right_on='toll_id')

In [ ]:
auti_tickets[auti_tickets['pobsubstatus']=='Rejected By Bank'].groupby('issue_identified').toll_fare_on_nhai_site.value_counts(normalize=True)*100   



In [ ]:
auti_tickets.toll_fare_on_nhai_site.value_counts(normalize=True)

In [ ]:
auti_tickets.groupby('issue_identified').toll_fare_on_nhai_site.value_counts(normalize=True)*100